# Statistical Evaluation

In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

from ddpg_td3_layernorm_lunar.ddpg import DDPG
from ddpg_td3_layernorm_lunar.replay_buffer import ReplayBuffer
from ddpg_td3_layernorm_lunar.run import run_policy
from ddpg_td3_layernorm_lunar.td3 import TD3
from scipy import stats
from statsmodels.stats.multitest import multipletests

%load_ext autoreload
%autoreload 2

In [3]:
def trimmed_stats(runs, trim=0.1):
    mean = np.full(runs.shape[1], np.nan)
    std = np.full(runs.shape[1], np.nan)
    for timestep, values in enumerate(runs.T):
        values = np.sort(values[~np.isnan(values)])
        if values.size == 0:
            continue
        n_trim = int(values.size * trim)
        if n_trim > 0:
            values = values[n_trim:-n_trim]
        mean[timestep] = values.mean()
        std[timestep] = values.std()
    return mean, std


def load_and_process_data(
    results_dir: Path,
    algo: type[DDPG] | type[TD3],
    layer_normalization: bool,
    seeds: range | list,
):
    seeds = list(seeds)

    steps = []
    evaluations = []
    biases = []

    for seed in seeds:
        path = results_dir / (
            f"{algo.__name__}_seed_{seed}_ln_{layer_normalization}.npz"
        )

        with np.load(path) as data:
            steps.append(data["steps"])
            evaluations.append(data["evaluations"])
            biases.append(data["biases"])

    steps = np.stack(steps)
    evaluations = np.stack(evaluations)
    biases = np.stack(biases)

    # Make sure all runs use the same evaluation steps
    if not np.all(steps == steps[0]):
        raise ValueError("Evaluation steps differ between runs.")

    mean_evaluations, std_evaluations = trimmed_stats(evaluations, trim=0.1)
    mean_biases, std_biases = trimmed_stats(biases, trim=0.1)

    return (
        steps[0],
        mean_evaluations,
        std_evaluations,
        mean_biases,
        std_biases,
    )


results_dir = Path("../results")

In [4]:
seed_metrics = {
    ("DDPG", False): {"bias": [], "return": []},
    ("DDPG", True):  {"bias": [], "return": []},
    ("TD3", False):  {"bias": [], "return": []},
    ("TD3", True):   {"bias": [], "return": []},
}

In [5]:
def run_full_statistical_analysis(results_dir: Path, seeds: list, window_size: int | None = 5):
    algos = [DDPG, TD3]
    ln_options = [False, True]
    seed_metrics = {
        (algo.__name__, ln): {'bias': [], 'return': []}
        for algo in algos for ln in ln_options
    }

    # window_size=None averages all checkpoints; otherwise use the final window.
    for seed in seeds:
        for algo in algos:
            for ln in ln_options:
                path = results_dir / f"{algo.__name__}_seed_{seed}_ln_{ln}.npz"
                if not path.exists():
                    print(f"Missing: {path.name}")
                    continue
                    
                with np.load(path) as data:
                    evaluations = data["evaluations"]
                    biases = data["biases"]
                    
                    # clean out NaN
                    valid_evals = evaluations[~np.isnan(evaluations)]
                    valid_biases = biases[~np.isnan(biases)]
                    
                    if len(valid_evals) == 0 or len(valid_biases) == 0:
                        print(f"Warning: All values are NaN for {path.name}")
                        continue
                    
                    selected_evals = valid_evals if window_size is None else valid_evals[-window_size:]
                    selected_biases = valid_biases if window_size is None else valid_biases[-window_size:]
                    final_return = np.mean(selected_evals)
                    final_bias = np.mean(selected_biases)
                    
                    seed_metrics[(algo.__name__, ln)]["bias"].append(final_bias)
                    seed_metrics[(algo.__name__, ln)]["return"].append(final_return)

    for key in seed_metrics:
        seed_metrics[key]["bias"] = np.array(seed_metrics[key]["bias"])
        seed_metrics[key]["return"] = np.array(seed_metrics[key]["return"])

    # pairwise comparisons
    comparisons = [
        (
            "TD3 vs DDPG (Standard / No LN)", 
            seed_metrics[("TD3", False)]["bias"], 
            seed_metrics[("DDPG", False)]["bias"], 
            "Bias"
        ),
        (
            "TD3 vs DDPG (with LayerNorm)", 
            seed_metrics[("TD3", True)]["bias"], 
            seed_metrics[("DDPG", True)]["bias"], 
            "Bias"
        ),
        (
            "LayerNorm effect on DDPG", 
            seed_metrics[("DDPG", True)]["bias"], 
            seed_metrics[("DDPG", False)]["bias"], 
            "Bias"
        ),
        (
            "LayerNorm effect on TD3", 
            seed_metrics[("TD3", True)]["bias"], 
            seed_metrics[("TD3", False)]["bias"], 
            "Bias"
        ),
    ]

    print("--- 1. PAIRED STATISTICAL TESTS (BIAS) ---")
    raw_p_values = []
    comparison_results = []

    for name, group_a, group_b, metric_name in comparisons:
        differences = group_a - group_b
        mean_diff = np.mean(differences)
        median_diff = np.median(differences)
        
        # 95% confidence interval using t-distribution on differences
        ci_low, ci_high = stats.t.interval(
            0.95, len(differences)-1, loc=mean_diff, scale=stats.sem(differences)
        )

        # Wilcoxon signed-rank test (non-parametric paired test)
        _, p_val = stats.wilcoxon(group_a, group_b)
        
        raw_p_values.append(p_val)
        comparison_results.append({
            "name": name,
            "mean_diff": mean_diff,
            "median_diff": median_diff,
            "ci": (ci_low, ci_high),
            "raw_p": p_val
        })

    # Holm's correction for multiple hypothesis testing
    rejected, corrected_p_values, _, _ = multipletests(raw_p_values, alpha=0.05, method='holm')

    for res, corr_p, rej in zip(comparison_results, corrected_p_values, rejected):
        print(f"\nComparison: {res['name']}")
        print(f"  Mean Difference: {res['mean_diff']:.3f} (Median: {res['median_diff']:.3f})")
        print(f"  95% CI: [{res['ci'][0]:.3f}, {res['ci'][1]:.3f}]")
        print(f"  Raw p-value: {res['raw_p']:.4f} | Holm-corrected p-value: {corr_p:.4f} (Significant: {rej})")

    print("\n--- 2. BIAS VS PERFORMANCE CORRELATION (Spearman) ---")
    for (algo_name, ln), data in seed_metrics.items():
        corr, p_corr = stats.spearmanr(data["bias"], data["return"])
        print(f"Config [{algo_name} | LN={ln}] -> Spearman rho: {corr:.3f} (p-value: {p_corr:.4f})")

## Statistical analysis over all checkpoints

Each seed contributes one value per condition, computed by averaging its complete checkpoint trajectory. Checkpoints are therefore not treated as independent observations.

In [6]:
run_full_statistical_analysis(
    results_dir,
    seeds=range(15),
    window_size=None,  # use every available checkpoint for each seed
)

--- 1. PAIRED STATISTICAL TESTS (BIAS) ---

Comparison: TD3 vs DDPG (Standard / No LN)
  Mean Difference: -55.868 (Median: -51.677)
  95% CI: [-68.936, -42.799]
  Raw p-value: 0.0001 | Holm-corrected p-value: 0.0002 (Significant: True)

Comparison: TD3 vs DDPG (with LayerNorm)
  Mean Difference: -190.086 (Median: -134.349)
  95% CI: [-300.073, -80.100]
  Raw p-value: 0.0001 | Holm-corrected p-value: 0.0002 (Significant: True)

Comparison: LayerNorm effect on DDPG
  Mean Difference: 136.614 (Median: 59.764)
  95% CI: [32.111, 241.118]
  Raw p-value: 0.0001 | Holm-corrected p-value: 0.0002 (Significant: True)

Comparison: LayerNorm effect on TD3
  Mean Difference: 2.396 (Median: 1.237)
  95% CI: [-8.133, 12.924]
  Raw p-value: 0.5614 | Holm-corrected p-value: 0.5614 (Significant: False)

--- 2. BIAS VS PERFORMANCE CORRELATION (Spearman) ---
Config [DDPG | LN=False] -> Spearman rho: -0.864 (p-value: 0.0000)
Config [DDPG | LN=True] -> Spearman rho: -0.893 (p-value: 0.0000)
Config [TD3 | LN